# DANA Solver

Runs DANA inference on all 174 benchmark instances (cordeau 28 + solomon 56 + gehring 6 + x_instances 3 = 93 actually from benchmark.py, but main.py has more solomon/gehring/x_instances). Output CSV is uploaded as `elfateh/dana-results-dana`.

In [ ]:
import os, subprocess, sys, glob, json, time

# Install deps
subprocess.run(["pip", "install", "-q", "pyyaml", "numpy", "scipy", "kagglehub", "requests", "tqdm"], check=True)
import torch
# Clone repo
REPO = "https://github.com/elfateh4/dana.git"
REPO_DIR = "/kaggle/working/dana"
if not os.path.exists(REPO_DIR):
    env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}
    subprocess.run(["git", "clone", "--depth", "1", REPO, REPO_DIR], check=True, env=env)
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "model"))

In [ ]:
import torch, numpy as np, yaml

# Download instances from Kaggle dataset or GitHub fallback
INSTANCE_DIR = "/kaggle/working/instances"
if not os.path.exists(INSTANCE_DIR):
    try:
        import kagglehub
        path = kagglehub.dataset_download("elfateh/dana-instances")
        subprocess.run(["cp", "-r", os.path.join(path, "*"), "/kaggle/working/instances/"], check=True)
    except Exception as e:
        print(f"Kaggle download failed, trying GitHub: {e}")
        import requests as _requests
        BASE_URL = "https://raw.githubusercontent.com/PyVRP/Instances/main"
        SETS = {
            "cordeau": ("MDVRPTW", [f"PR{i}{s}.vrp" for i in range(11, 25) for s in ("A", "B")]),
            "solomon": ("VRPTW/Solomon", ["C101.vrp","C102.vrp","C103.vrp","C104.vrp","C105.vrp","C106.vrp","C107.vrp","C108.vrp","C109.vrp","C201.vrp","C202.vrp","C203.vrp","C204.vrp","C205.vrp","C206.vrp","C207.vrp","C208.vrp","R101.vrp","R102.vrp","R103.vrp","R104.vrp","R105.vrp","R106.vrp","R107.vrp","R108.vrp","R109.vrp","R110.vrp","R111.vrp","R112.vrp","R201.vrp","R202.vrp","R203.vrp","R204.vrp","R205.vrp","R206.vrp","R207.vrp","R208.vrp","R209.vrp","R210.vrp","R211.vrp","RC101.vrp","RC102.vrp","RC103.vrp","RC104.vrp","RC105.vrp","RC106.vrp","RC107.vrp","RC108.vrp","RC201.vrp","RC202.vrp","RC203.vrp","RC204.vrp","RC205.vrp","RC206.vrp","RC207.vrp","RC208.vrp"]),
            "gehring": ("VRPTW", [f"{t}{n}_10_{i}.vrp" for t in ("C","R","RC") for n in (1,2) for i in range(1,11)]),
            "x_instances": ("CVRP", ["X-n101-k25.vrp","X-n106-k14.vrp","X-n110-k13.vrp","X-n115-k10.vrp","X-n120-k6.vrp","X-n125-k30.vrp","X-n129-k18.vrp","X-n134-k13.vrp","X-n139-k10.vrp","X-n143-k7.vrp","X-n153-k22.vrp","X-n157-k13.vrp","X-n162-k11.vrp","X-n167-k10.vrp","X-n176-k26.vrp","X-n181-k23.vrp","X-n186-k15.vrp","X-n190-k8.vrp","X-n195-k51.vrp","X-n200-k36.vrp","X-n204-k19.vrp","X-n209-k16.vrp","X-n214-k11.vrp","X-n219-k73.vrp","X-n223-k34.vrp","X-n228-k23.vrp","X-n233-k16.vrp","X-n237-k14.vrp","X-n242-k48.vrp","X-n247-k50.vrp"]),
        }
        for sname, (rdir, files) in SETS.items():
            dest = os.path.join(INSTANCE_DIR, sname)
            os.makedirs(dest, exist_ok=True)
            for fname in files:
                url = f"{BASE_URL}/{rdir}/{fname}"
                path_d = os.path.join(dest, fname)
                if not os.path.exists(path_d):
                    try:
                        r = _requests.get(url, timeout=30)
                        if r.status_code == 200:
                            with open(path_d, "w") as f:
                                f.write(r.text)
                    except Exception:
                        pass
        print("Downloaded instances from GitHub")

# Load checkpoint
KAGGLE_CKPT_DIR = "/kaggle/input/dana-checkpoints"
CKPT_DIR = KAGGLE_CKPT_DIR if os.path.exists(KAGGLE_CKPT_DIR) else "/kaggle/working/checkpoints_download"
if not os.path.exists(CKPT_DIR) or not glob.glob(os.path.join(CKPT_DIR, "*.pt")):
    os.makedirs(CKPT_DIR, exist_ok=True)
    try:
        import kagglehub
        path = kagglehub.dataset_download("elfateh/dana-checkpoints")
        for f in glob.glob(os.path.join(path, "*.pt")):
            subprocess.run(["cp", f, CKPT_DIR], check=True)
    except Exception as e:
        print(f"Failed to download checkpoints: {e}")

ckpt_files = sorted(glob.glob(os.path.join(CKPT_DIR, "*.pt")))
print(f"Checkpoints found: {len(ckpt_files)}")

with open("configs/dana.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["model"]["num_encoder_layers"] = 4
cfg["model"]["num_decoder_layers"] = 2
cfg["model"]["feedforward_dim"] = 256
cfg["data"]["num_locations"] = 50

device = "cuda" if torch.cuda.is_available() else "cpu"
# Fallback to CPU if CUDA fails
if device == "cuda":
    try:
        torch.zeros(1, device="cuda")
    except Exception:
        device = "cpu"
        print("CUDA not available, using CPU")
print(f"Device: {device}")

if ckpt_files:
    from train import build_policy
    dana_policy = build_policy(cfg).to(device)
    dana_policy.eval()
    ckpt = torch.load(ckpt_files[-1], map_location=device)
    dana_policy.load_state_dict(ckpt["policy_state_dict"])
    print(f"Loaded checkpoint: {ckpt_files[-1]}")
else:
    dana_policy = None
    print("No checkpoint found — will skip DANA")

In [ ]:
def parse_vrp_coords(path):
    coords, section = [], False
    with open(path) as f:
        for line in f:
            s = line.strip()
            if s.upper().startswith("NODE_COORD_SECTION"):
                section = True
                continue
            if any(kw in s.upper() for kw in ["DEMAND_SECTION", "DEPOT_SECTION", "TIME_WINDOW_SECTION", "EOF"]):
                section = False
                continue
            if section and s:
                parts = s.split()
                if len(parts) >= 3:
                    coords.append((float(parts[1]), float(parts[2])))
    return np.array(coords, dtype=np.float32)


def parse_num_depots(path):
    depots = []
    in_depot = False
    with open(path) as f:
        for line in f:
            s = line.strip()
            if s.upper().startswith("DEPOT_SECTION"):
                in_depot = True
                continue
            if s.upper().startswith("EOF"):
                break
            if in_depot and s:
                depots.append(int(s))
    return len(depots)


def run_dana_on_instance(policy, vrp_path, device, cfg):
    """
    Optimized DANA inference: caches encoder output to avoid re-encoding per step.
    Returns routes as 0-based customer indices + raw Euclidean cost.
    """
    coords_np = parse_vrp_coords(vrp_path)
    N = len(coords_np)
    B = 1
    coords_t = torch.tensor(coords_np, dtype=torch.float, device=device).unsqueeze(0)
    diff = coords_np[:, None] - coords_np[None, :]
    dist_np = np.sqrt((diff**2).sum(axis=-1))
    dist_mat_t = torch.tensor(dist_np, dtype=torch.float, device=device).unsqueeze(0)

    num_depots = parse_num_depots(vrp_path)
    depot_mask = torch.zeros(B, N, dtype=torch.bool, device=device)
    depot_mask[:, :num_depots] = True
    demand = torch.ones(B, N, dtype=torch.float, device=device)
    tw_start = torch.zeros(B, N, dtype=torch.float, device=device)
    tw_end = torch.full((B, N), 480.0, dtype=torch.float, device=device)
    visited = depot_mask.clone()

    # Cache encoder + context module output
    with torch.no_grad():
        node_emb = policy.encoder(coords_t, dist_mat_t)
        B_enc, N_enc, D = node_emb.shape
        context = torch.zeros(B_enc, D, device=node_emb.device)
        node_emb, context = policy.context_module(node_emb, dist_mat_t, context)

    actions = []
    with torch.no_grad():
        for _ in range(cfg["environment"]["max_vehicles"]):
            if visited[:, num_depots:].all():
                break
            for _ in range(N * 2):
                visit_frac = visited.float().mean(dim=-1)
                remaining_cap = 1.0 - (demand.float().mean(dim=-1) * visit_frac)
                unvisited_tw_end = tw_end.float().masked_fill(visited, float("inf"))
                min_tw_end = unvisited_tw_end.min(dim=-1).values
                max_tw_end = unvisited_tw_end.max(dim=-1).values
                tw_urgency = 1.0 - (min_tw_end / (max_tw_end + 1e-8))
                unvisited_count = (~visited).float().sum(dim=-1) / N
                vehicle_state = torch.stack(
                    [visit_frac, remaining_cap, tw_urgency, unvisited_count], dim=-1
                )
                vehicle_feat = policy.vehicle_embedding(vehicle_state)

                mask = visited.clone()
                logits = policy.decoder(node_emb, context, vehicle_feat, mask=mask, num_starts=1)
                logits = logits.masked_fill(visited, float("-inf"))
                a = logits.argmax(dim=-1)
                actions.append(a)
                visited[:, a.item()] = True
                visited[:, :num_depots] = depot_mask[:, :num_depots]
                if visited[:, num_depots:].all():
                    break

    if len(actions) < 2:
        return {"routes": [], "raw_euclidean_cost": 0.0, "status": "no_solution"}

    acts = torch.cat(actions)

    # Split by depot visits to get routes
    routes = []
    current = []
    for idx in acts.tolist():
        if idx < num_depots:
            if current:
                routes.append(current)
                current = []
        else:
            current.append(idx)
    if current:
        routes.append(current)

    # Compute raw Euclidean cost with best depot assignment
    total_cost = 0.0
    for route in routes:
        if not route:
            continue
        first, last = route[0], route[-1]
        best_depot = min(range(num_depots), key=lambda d: dist_np[d, first] + dist_np[last, d])
        total_cost += dist_np[best_depot, first]
        for i in range(len(route) - 1):
            total_cost += dist_np[route[i], route[i + 1]]
        total_cost += dist_np[last, best_depot]

    return {"routes": routes, "raw_euclidean_cost": total_cost, "status": "success"}

In [ ]:
import csv

BENCHMARKS = {
    "cordeau": {"dir": "cordeau", "problem_type": "mdvrptw"},
    "solomon": {"dir": "solomon", "problem_type": "vrptw"},
    "gehring": {"dir": "gehring", "problem_type": "vrptw"},
    "x_instances": {"dir": "x_instances", "problem_type": "cvrp"},
}

OUT_CSV = "/kaggle/working/dana_results.csv"
rows = []

if dana_policy is None:
    print("No DANA checkpoint — nothing to run")
else:
    for bench_name, bench_info in BENCHMARKS.items():
        bench_dir = os.path.join(INSTANCE_DIR, bench_info["dir"])
        if not os.path.exists(bench_dir):
            print(f"SKIP: {bench_dir} not found")
            continue
        instance_files = sorted([f for f in os.listdir(bench_dir) if f.endswith(".vrp")])
        print(f"\n=== {bench_name} ({len(instance_files)} instances) ===")

        for fname in instance_files:
            inst_file = os.path.join(bench_dir, fname)
            inst_name = fname.replace(".vrp", "")
            t0 = time.time()
            try:
                result = run_dana_on_instance(dana_policy, inst_file, device, cfg)
                elapsed = time.time() - t0
                rows.append({
                    "instance": inst_name,
                    "solver": "dana",
                    "benchmark": bench_name,
                    "problem_type": bench_info["problem_type"],
                    "cost": result["raw_euclidean_cost"],
                    "raw_euclidean_cost": result["raw_euclidean_cost"],
                    "cost_unit": "euclidean",
                    "time_s": round(elapsed, 2),
                    "status": result["status"],
                    "error_msg": "",
                })
                print(f"  {inst_name}: {result['raw_euclidean_cost']:.2f} ({elapsed:.1f}s)")
            except Exception as e:
                elapsed = time.time() - t0
                rows.append({
                    "instance": inst_name,
                    "solver": "dana",
                    "benchmark": bench_name,
                    "problem_type": bench_info["problem_type"],
                    "cost": None,
                    "raw_euclidean_cost": None,
                    "cost_unit": "euclidean",
                    "time_s": round(elapsed, 2),
                    "status": "error",
                    "error_msg": str(e),
                })
                print(f"  {inst_name}: ERROR {e}")

with open(OUT_CSV, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["instance", "solver", "benchmark", "problem_type",
                                            "cost", "raw_euclidean_cost", "cost_unit", "time_s",
                                            "status", "error_msg"])
    writer.writeheader()
    writer.writerows(rows)

print(f"\nSaved {len(rows)} results to {OUT_CSV}")

In [ ]:
DATASET_DIR = "/kaggle/working/dana-results-dana"
os.makedirs(DATASET_DIR, exist_ok=True)
subprocess.run(["cp", OUT_CSV, os.path.join(DATASET_DIR, "dana_results.csv")], check=True)

with open(os.path.join(DATASET_DIR, "dataset-metadata.json"), "w") as f:
    json.dump({
        "title": "DANA Results",
        "id": "elfateh/dana-results-dana",
        "licenses": [{"name": "CC0-1.0"}],
    }, f)

# Always try version first (dataset already exists), fall back to create
r = subprocess.run(
    ["kaggle", "datasets", "version", "-p", DATASET_DIR, "-m", "Updated results", "--dir-mode", "zip"],
    capture_output=True, text=True,
)
if r.returncode != 0:
    r2 = subprocess.run(
        ["kaggle", "datasets", "create", "-p", DATASET_DIR, "--dir-mode", "zip"],
        capture_output=True, text=True,
    )
    if r2.returncode != 0:
        print(f"Dataset upload failed: {r2.stderr.strip()}")
    else:
        print("Dataset created successfully.")
else:
    print("Dataset version updated successfully.")